# Stratégie 5 — Elastic Weight Consolidation (Food101)

Ce notebook implémente **EWC (Kirkpatrick et al., 2017)** pour limiter l'oubli catastrophique lors du passage BF→HF.

## Pipeline
1. **Phase A — Pré-entraînement BF** (10 époques, LR=1e-3) : identique à la Stratégie 1.
2. **Ancrage EWC** : calcul de la diagonale de la Fisher Information Matrix sur les données BF.  
   Les poids `θ*` (optimaux BF) sont enregistrés.
3. **Phase B — Fine-tuning HF + pénalité EWC** (10 époques, LR=3e-4) :
   $$\mathcal{L}_{\text{total}} = \mathcal{L}_{CE}(HF) + \lambda \sum_i F_i (\theta_i - \theta_i^*)^2$$

**Objectif** : retrouver la performance HF de la Stratégie 1 tout en limitant la chute de robustesse BF  
(le *domain forgetting* identifié comme limite principale de la Stratégie 1).

In [ ]:
import os, json, time, random, statistics
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import wandb
    _WANDB_AVAILABLE = True
except ImportError:
    _WANDB_AVAILABLE = False
    print('wandb non installe')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil: {device}')

import sys, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import degradation; importlib.reload(degradation)
from degradation import DegradedDataset, hf_transform, clean_tensor_transform
import cost; importlib.reload(cost)
from cost import data_cost
import env_config; importlib.reload(env_config)
print('Modules charges.')

In [ ]:
DATASET_NAME     = 'Food101'
BASE_DIR         = env_config.ensure_dataset_ready(DATASET_NAME)
RESULTS_DIR      = env_config.results_dir(DATASET_NAME)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Dataset    :', DATASET_NAME)
print('BASE_DIR   :', BASE_DIR)
print('RESULTS_DIR:', RESULTS_DIR)

COST_HF          = 10
COST_BF          = 1
BATCH_SIZE       = 64
NUM_WORKERS      = min(8, os.cpu_count() or 2)
PRETRAIN_EPOCHS  = 10
FINETUNE_EPOCHS  = 10
LR_PRETRAIN      = 1e-3
LR_FINETUNE      = 3e-4
HF_TRAIN_RATIO   = 0.8
EWC_LAMBDA       = 400    # Poids de la penalite EWC (hyperparametre cle, defaut: 400)
N_FISHER_SAMPLES = 1000   # Nb images BF pour l'approximation de la Fisher diagonale
SEEDS            = [42, 1, 2]
USE_WANDB        = True
WANDB_PROJECT    = 'PF22-MultiFidelity'
print(f'EWC_LAMBDA={EWC_LAMBDA} | N_FISHER_SAMPLES={N_FISHER_SAMPLES}')

In [ ]:
for req in ['train/BF', 'train/HF', 'test']:
    p = os.path.join(BASE_DIR, req)
    if not os.path.isdir(p): raise FileNotFoundError(f'Dossier manquant : {p}')

transform_hf    = hf_transform()
transform_clean = clean_tensor_transform()

dataset_hf_full  = datasets.ImageFolder(os.path.join(BASE_DIR, 'train/HF'), transform=transform_hf)
dataset_bf_train = DegradedDataset(
    datasets.ImageFolder(os.path.join(BASE_DIR, 'train/BF'), transform=transform_clean), seeded=True)
dataset_test_hf  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_hf)
dataset_test_bf  = DegradedDataset(
    datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_clean), seeded=True)

NUM_CLASSES = len(dataset_hf_full.classes)
print(f'Classes ({NUM_CLASSES}): {dataset_hf_full.classes}')
print(f'HF pool: {len(dataset_hf_full)} | BF train: {len(dataset_bf_train)} | Test: {len(dataset_test_hf)}')

In [ ]:
def create_model(num_classes=10):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(device)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = total_correct = total_n = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); loss = criterion(logits, y)
            total_loss    += loss.item() * x.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_n       += x.size(0)
    return total_loss / total_n, 100.0 * total_correct / total_n

def make_grad_scaler():
    try:    return GradScaler('cuda', enabled=(device.type == 'cuda'))
    except TypeError: return GradScaler(enabled=(device.type == 'cuda'))

def train_phase(model, train_loader, val_loader, epochs, lr, phase_name,
                use_wandb=False, global_epoch_offset=0):
    """Entrainement standard (CE loss seule). Utilise pour le pre-entrainement BF."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler    = make_grad_scaler()
    hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    t0 = time.time(); log_every = 10

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0; ep_n = 0
        ep_start = time.time()
        for bi, (x, y) in enumerate(train_loader, 1):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            ep_loss += loss.item() * x.size(0); ep_n += x.size(0)
            if bi % log_every == 0 or bi == 1 or bi == len(train_loader):
                elapsed = time.time() - ep_start
                bps = bi / elapsed if elapsed > 0 else 0.0
                eta = (len(train_loader) - bi) / bps / 60.0 if bps > 0 else 0.0
                print(f'[{phase_name}] Ep {epoch}/{epochs} | Batch {bi}/{len(train_loader)} | '
                      f'loss={loss.item():.4f} | {bps:.1f} b/s | ETA={eta:.1f} min')
        train_loss = ep_loss / ep_n
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        ep_min = (time.time() - ep_start) / 60.0
        hist['train_loss'].append(train_loss); hist['val_loss'].append(val_loss)
        hist['val_acc'].append(val_acc)
        print(f'[{phase_name}] Ep {epoch}/{epochs} : '
              f'train={train_loss:.4f} val={val_loss:.4f} acc={val_acc:.2f}% | {ep_min:.2f} min')
        if use_wandb and _WANDB_AVAILABLE:
            wandb.log({'epoch': global_epoch_offset + epoch, 'phase': phase_name,
                       'train/loss': train_loss, 'val/loss': val_loss, 'val/accuracy': val_acc})
    elapsed_min = (time.time() - t0) / 60.0
    print(f'[{phase_name}] Termine en {elapsed_min:.2f} min')
    return hist, elapsed_min

print('Utilitaires modele et train_phase definis.')

In [ ]:
def compute_fisher_diagonal(model, loader, n_samples=1000):
    """Calcule la diagonale de la Fisher Information Matrix (approximation empirique).

    FIM_diag_i = E[(d/dtheta_i * log p(y|x,theta))^2]
    Approche   : moyenne des carres des gradients de NLL sur n_samples images BF.
    Calcul en  float32 (sans autocast) pour stabilite numerique.
    """
    fisher = {n: torch.zeros_like(p.data)
              for n, p in model.named_parameters() if p.requires_grad}
    model.eval()
    count = 0

    for x, y in loader:
        if count >= n_samples:
            break
        x, y = x.to(device), y.to(device)
        model.zero_grad()
        with torch.enable_grad():
            logits    = model(x).float()
            log_probs = F.log_softmax(logits, dim=1)
            loss      = F.nll_loss(log_probs, y)
            loss.backward()
        bs = x.size(0)
        for n, p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                fisher[n].data += p.grad.float().data.pow(2) * bs
        count += bs

    for n in fisher:
        fisher[n] /= max(count, 1)

    all_vals = torch.cat([f.flatten() for f in fisher.values()])
    print(f'Fisher diagonale calculee sur {count} images.')
    print(f'Fisher stats : mean={all_vals.mean():.2e} | max={all_vals.max():.2e} | '
          f'non-zero={(all_vals > 1e-15).float().mean()*100:.1f}%')
    return fisher

print('Fonction compute_fisher_diagonal definie.')

In [ ]:
def ewc_penalty(model, fisher, params_star, ewc_lambda):
    """Penalite EWC : lambda * sum_i F_i * (theta_i - theta_i*)^2"""
    penalty = torch.tensor(0.0, device=device)
    for n, p in model.named_parameters():
        if p.requires_grad and n in fisher and n in params_star:
            diff    = p.float() - params_star[n].float()
            penalty = penalty + (fisher[n].float() * diff.pow(2)).sum()
    return ewc_lambda * penalty

def train_finetune_ewc(model, train_loader, val_loader, epochs, lr, phase_name,
                       fisher, params_star, ewc_lambda,
                       use_wandb=False, global_epoch_offset=0):
    """Fine-tuning HF avec penalite EWC.

    Loss totale : L_CE(HF) + lambda * sum_i F_i (theta_i - theta_i*)^2
    La penalite EWC est calculee en float32 (hors autocast) pour stabilite.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler    = make_grad_scaler()
    hist = {'train_loss': [], 'train_ce': [], 'train_ewc': [], 'val_loss': [], 'val_acc': []}
    t0 = time.time(); log_every = 10

    for epoch in range(1, epochs + 1):
        model.train()
        ep_total = 0.0; ep_ce = 0.0; ep_ewc_v = 0.0; ep_n = 0
        ep_start = time.time()

        for bi, (x, y) in enumerate(train_loader, 1):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits  = model(x)
                ce_loss = criterion(logits, y)

            # Penalite EWC en float32, hors autocast
            ewc_val    = ewc_penalty(model, fisher, params_star, ewc_lambda)
            total_loss = ce_loss.float() + ewc_val
            scaler.scale(total_loss).backward()
            scaler.step(optimizer); scaler.update()

            bs = x.size(0)
            ep_total  += total_loss.item() * bs
            ep_ce     += ce_loss.item()    * bs
            ep_ewc_v  += ewc_val.item()    * bs
            ep_n      += bs

            if bi % log_every == 0 or bi == 1 or bi == len(train_loader):
                elapsed = time.time() - ep_start
                bps = bi / elapsed if elapsed > 0 else 0.0
                eta = (len(train_loader) - bi) / bps / 60.0 if bps > 0 else 0.0
                print(f'[{phase_name}] Ep {epoch}/{epochs} | '
                      f'Batch {bi}/{len(train_loader)} | '
                      f'total={total_loss.item():.4f} CE={ce_loss.item():.4f} '
                      f'EWC={ewc_val.item():.4f} | ETA={eta:.1f} min')

        train_loss = ep_total / ep_n; train_ce = ep_ce / ep_n; train_ewc = ep_ewc_v / ep_n
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        ep_min = (time.time() - ep_start) / 60.0

        hist['train_loss'].append(train_loss); hist['train_ce'].append(train_ce)
        hist['train_ewc'].append(train_ewc);   hist['val_loss'].append(val_loss)
        hist['val_acc'].append(val_acc)

        print(f'[{phase_name}] Ep {epoch}/{epochs} : '
              f'total={train_loss:.4f} CE={train_ce:.4f} EWC={train_ewc:.4f} | '
              f'val_acc={val_acc:.2f}% | {ep_min:.2f} min')

        if use_wandb and _WANDB_AVAILABLE:
            wandb.log({'epoch': global_epoch_offset + epoch, 'phase': phase_name,
                       'train/total_loss': train_loss, 'train/ce_loss': train_ce,
                       'train/ewc_penalty': train_ewc,
                       'val/loss': val_loss, 'val/accuracy': val_acc})

    elapsed_min = (time.time() - t0) / 60.0
    print(f'[{phase_name}] Termine en {elapsed_min:.2f} min')
    return hist, elapsed_min

print('Fonctions ewc_penalty et train_finetune_ewc definies.')

In [ ]:
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def run_seed(seed):
    set_all_seeds(seed)
    n_hf = int(HF_TRAIN_RATIO * len(dataset_hf_full))
    ds_hf_train, ds_hf_val = random_split(
        dataset_hf_full, [n_hf, len(dataset_hf_full) - n_hf],
        generator=torch.Generator().manual_seed(seed))

    ld_bf_train = DataLoader(dataset_bf_train, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True)
    ld_hf_train = DataLoader(ds_hf_train, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True)
    ld_hf_val   = DataLoader(ds_hf_val,   batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    ld_test_hf  = DataLoader(dataset_test_hf, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    ld_test_bf  = DataLoader(dataset_test_bf, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    track = USE_WANDB and _WANDB_AVAILABLE
    if track:
        wandb.init(project=WANDB_PROJECT,
                   name=f'Strategie5_EWC_{DATASET_NAME}_seed{seed}',
                   tags=['strategy5', 'ewc', DATASET_NAME, f'seed_{seed}'],
                   reinit=True,
                   config={'strategy': 'ewc', 'dataset': DATASET_NAME, 'seed': seed,
                           'pretrain_epochs': PRETRAIN_EPOCHS, 'finetune_epochs': FINETUNE_EPOCHS,
                           'lr_pretrain': LR_PRETRAIN, 'lr_finetune': LR_FINETUNE,
                           'ewc_lambda': EWC_LAMBDA, 'n_fisher_samples': N_FISHER_SAMPLES,
                           'batch_size': BATCH_SIZE, 'architecture': 'resnet18'})

    # --- Phase A : Pre-entrainement BF ---
    model = create_model(num_classes=NUM_CLASSES)
    hist_pre, t_pre = train_phase(model, ld_bf_train, ld_hf_val, PRETRAIN_EPOCHS,
                                  LR_PRETRAIN, 'Pretrain_BF',
                                  use_wandb=track, global_epoch_offset=0)

    # --- Ancrage EWC : Fisher + parametres BF optimaux ---
    print(f'\nCalcul de la Fisher diagonale (lambda={EWC_LAMBDA}, n={N_FISHER_SAMPLES})...')
    fisher      = compute_fisher_diagonal(model, ld_bf_train, n_samples=N_FISHER_SAMPLES)
    params_star = {n: p.data.clone()
                   for n, p in model.named_parameters() if p.requires_grad}
    print(f'Parametres BF ancres ({len(params_star)} tenseurs).')

    # --- Phase B : Fine-tuning HF + penalite EWC ---
    hist_ft, t_ft = train_finetune_ewc(
        model, ld_hf_train, ld_hf_val, FINETUNE_EPOCHS, LR_FINETUNE,
        'Finetune_HF+EWC', fisher, params_star, EWC_LAMBDA,
        use_wandb=track, global_epoch_offset=PRETRAIN_EPOCHS)

    # --- Evaluation finale ---
    crit = nn.CrossEntropyLoss()
    _, hf_acc = evaluate(model, ld_test_hf, crit)
    _, bf_acc = evaluate(model, ld_test_bf, crit)
    mix_acc   = (hf_acc + bf_acc) / 2.0

    cost_bf = len(dataset_bf_train) * COST_BF * PRETRAIN_EPOCHS
    cost_hf = n_hf * COST_HF * FINETUNE_EPOCHS
    total_cost = float(cost_bf + cost_hf)

    if track:
        wandb.log({'test/accuracy_HF': hf_acc, 'test/accuracy_BF': bf_acc,
                   'test/accuracy_Mixte': mix_acc})
        wandb.finish()
    print(f'[seed {seed}] HF={hf_acc:.2f}% BF={bf_acc:.2f}% Mixte={mix_acc:.2f}% | '
          f'cost={total_cost:.0f} CA | {t_pre + t_ft:.1f} min')
    return {'seed': seed, 'accuracy_HF': hf_acc, 'accuracy_BF': bf_acc,
            'accuracy_Mixte': mix_acc, 'total_cost_CA': total_cost,
            'time_min': t_pre + t_ft, 'n_hf_train': n_hf,
            'hist_pre': hist_pre, 'hist_ft': hist_ft, 'model': model}

per_seed = [run_seed(s) for s in SEEDS]
model    = per_seed[0]['model']
hist_pre = per_seed[0]['hist_pre']
hist_ft  = per_seed[0]['hist_ft']

In [ ]:
def _agg(key):
    vals = [r[key] for r in per_seed]
    m = statistics.mean(vals); s = statistics.stdev(vals) if len(vals) > 1 else 0.0
    return m, s, vals

hf_m, hf_s, hf_all = _agg('accuracy_HF')
bf_m, bf_s, bf_all = _agg('accuracy_BF')
mx_m, mx_s, mx_all = _agg('accuracy_Mixte')
tt_m, tt_s, _      = _agg('time_min')
cost_m, _, _       = _agg('total_cost_CA')

n_hf_pool = len(dataset_hf_full); n_bf_pool = len(dataset_bf_train)
data_cost_CA = data_cost(n_hf=n_hf_pool, n_bf=n_bf_pool)

results = {
    'strategy': 'elastic_weight_consolidation',
    'dataset': DATASET_NAME,
    'pipeline': 'BF_pretraining_then_HF_finetuning_EWC',
    'pretrain_epochs': PRETRAIN_EPOCHS, 'finetune_epochs': FINETUNE_EPOCHS,
    'lr_pretrain': LR_PRETRAIN, 'lr_finetune': LR_FINETUNE,
    'ewc_lambda': EWC_LAMBDA, 'n_fisher_samples': N_FISHER_SAMPLES,
    'seeds': SEEDS, 'n_hf_pool': n_hf_pool, 'n_bf_pool': n_bf_pool,
    'data_cost_CA': float(data_cost_CA), 'total_cost_CA': cost_m,
    'accuracy_HF':    hf_m, 'accuracy_HF_std':    hf_s, 'accuracy_HF_seeds':    hf_all,
    'accuracy_BF':    bf_m, 'accuracy_BF_std':    bf_s, 'accuracy_BF_seeds':    bf_all,
    'accuracy_Mixte': mx_m, 'accuracy_Mixte_std': mx_s, 'accuracy_Mixte_seeds': mx_all,
    'total_time_min': tt_m, 'total_time_min_std': tt_s,
    'per_seed': [{k: v for k, v in r.items()
                  if k not in ('model', 'hist_pre', 'hist_ft')} for r in per_seed],
}

json_path = os.path.join(RESULTS_DIR, 'results_strategy5_ewc.json')
pth_path  = os.path.join(RESULTS_DIR, 'model_strategy5_ewc.pth')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
torch.save(model.state_dict(), pth_path)

print(f'--- RESULTATS ({len(SEEDS)} seeds) - {DATASET_NAME} - Strategie 5 (EWC) ---')
print(f'Test HF    : {hf_m:.2f} +/- {hf_s:.2f} %')
print(f'Test BF    : {bf_m:.2f} +/- {bf_s:.2f} %')
print(f'Test Mixte : {mx_m:.2f} +/- {mx_s:.2f} %')
print(f'Cout total : {cost_m:.0f} CA | lambda EWC = {EWC_LAMBDA}')
print('JSON :', json_path)

In [ ]:
x_pre = np.arange(1, PRETRAIN_EPOCHS + 1)
x_ft  = np.arange(PRETRAIN_EPOCHS + 1, PRETRAIN_EPOCHS + FINETUNE_EPOCHS + 1)
transition = PRETRAIN_EPOCHS + 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Strategie 5 (EWC: BF->HF) - {DATASET_NAME} (seed {SEEDS[0]})', fontsize=14)

for ax in axes:
    ax.axvspan(0.5, transition, alpha=0.08, color='#55A868')
    ax.axvspan(transition, PRETRAIN_EPOCHS + FINETUNE_EPOCHS + 0.5, alpha=0.08, color='#4C72B0')
    ax.axvline(transition, color='gray', linestyle=':', linewidth=1.2)

axes[0].plot(x_pre, hist_pre['train_loss'], marker='o', markersize=4,
             label='Train loss BF', color='tab:green')
axes[0].plot(x_pre, hist_pre['val_loss'],   marker='o', markersize=4, linestyle='--',
             label='Val loss HF (pre-train)', color='tab:olive')
axes[0].plot(x_ft,  hist_ft['train_ce'],    marker='o', markersize=4,
             label='Train CE loss HF', color='tab:blue')
axes[0].plot(x_ft,  hist_ft['train_ewc'],   marker='s', markersize=4, linestyle=':',
             label='EWC penalty', color='tab:red')
axes[0].plot(x_ft,  hist_ft['val_loss'],    marker='o', markersize=4, linestyle='--',
             label='Val loss HF (fine-tune)', color='tab:orange')
axes[0].set_title('Evolution des losses'); axes[0].set_xlabel('Epoch globale')
axes[0].set_ylabel('Loss'); axes[0].grid(alpha=0.3)

phase_patches = [
    mpatches.Patch(color='#55A868', alpha=0.4, label='Phase BF (pre-train)'),
    mpatches.Patch(color='#4C72B0', alpha=0.4, label='Phase HF + EWC'),
]
handles0, labels0 = axes[0].get_legend_handles_labels()
axes[0].legend(handles=handles0 + phase_patches, fontsize=8)

axes[1].plot(x_pre, hist_pre['val_acc'], marker='o', markersize=4,
             label='Val acc HF - Pre-train', color='tab:olive')
axes[1].plot(x_ft,  hist_ft['val_acc'],  marker='o', markersize=4,
             label='Val acc HF - EWC fine-tune', color='tab:blue')
axes[1].set_title('Precision validation HF'); axes[1].set_xlabel('Epoch globale')
axes[1].set_ylabel('Accuracy (%)'); axes[1].grid(alpha=0.3)
handles1, labels1 = axes[1].get_legend_handles_labels()
axes[1].legend(handles=handles1 + phase_patches, fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.95])
png_path = os.path.join(RESULTS_DIR, 'strategy5_ewc_curves.png')
plt.savefig(png_path, dpi=180, bbox_inches='tight')
plt.show()
print('Figure :', png_path)

In [ ]:
comparison_files = {
    'BL 1(HF)':             'results_baseline_HF.json',
    'BL 2(BF)':             'results_baseline_BF.json',
    'BL 3(MIXTE)':          'results_baseline_MIXTE.json',
    'Strat 1(BF->HF)':      'results_strategy1_transfer_learning.json',
    'Strat 2(CoTrain)':     'results_strategy2_cotraining_reweighting.json',
    'Strat 3(Curriculum)':  'results_strategy3_curriculum_learning.json',
    'Strat 5(EWC)':         'results_strategy5_ewc.json',
}
palette = {
    'BL 1(HF)':            '#4C72B0', 'BL 2(BF)':             '#55A868',
    'BL 3(MIXTE)':         '#C44E52', 'Strat 1(BF->HF)':      '#8172B2',
    'Strat 2(CoTrain)':    '#CCB974', 'Strat 3(Curriculum)':  '#64B5CD',
    'Strat 5(EWC)':        '#E07070',
}

def _time_min(d):
    for k in ('training_time_min', 'total_time_min'):
        if k in d: return float(d[k])
    if 'training_time_sec' in d: return d['training_time_sec'] / 60.0
    return float('nan')

loaded = {}
for name, fn in comparison_files.items():
    p = os.path.join(RESULTS_DIR, fn)
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f: loaded[name] = json.load(f)

if not loaded:
    print('Aucun JSON dans', RESULTS_DIR)
else:
    names     = list(loaded.keys())
    acc_hf    = [loaded[n].get('accuracy_HF',    float('nan')) for n in names]
    acc_bf    = [loaded[n].get('accuracy_BF',    float('nan')) for n in names]
    acc_mixte = [loaded[n].get('accuracy_Mixte', float('nan')) for n in names]
    costs     = [loaded[n].get('total_cost_CA',  float('nan')) for n in names]
    times     = [_time_min(loaded[n]) for n in names]
    colors    = [palette.get(n, '#888888') for n in names]

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f'Comparaison Baselines + Strategies — {DATASET_NAME}', fontsize=14)

    x = np.arange(3); width = 0.10; n = len(names)
    offsets = (np.arange(n) - (n - 1) / 2.0) * width
    for i, name in enumerate(names):
        vals = [acc_hf[i], acc_bf[i], acc_mixte[i]]
        bars = axes[0].bar(x + offsets[i], vals, width=width,
                           label=name, color=palette.get(name, '#888888'))
        for bar in bars:
            h = bar.get_height()
            if h == h:
                axes[0].text(bar.get_x() + bar.get_width() / 2, h + 0.4,
                             f'{h:.1f}', ha='center', va='bottom', fontsize=6)
    axes[0].set_title('Precisions'); axes[0].set_xticks(x)
    axes[0].set_xticklabels(['Test HF', 'Test BF', 'Test Mixte'])
    axes[0].set_ylabel('Accuracy (%)'); axes[0].set_ylim(0, 100)
    axes[0].grid(axis='y', alpha=0.3); axes[0].legend(fontsize=7, ncol=2)

    bars = axes[1].bar(names, costs, color=colors)
    for bar, v in zip(bars, costs):
        if v == v:
            axes[1].text(bar.get_x() + bar.get_width() / 2, v,
                         f"{int(v):,}".replace(',', ' '), ha='center', va='bottom', fontsize=8)
    axes[1].set_title('Cout total estimé'); axes[1].set_ylabel('CA')
    vc = [c for c in costs if c == c]
    if vc: axes[1].set_ylim(0, max(vc) * 1.15)
    axes[1].grid(axis='y', alpha=0.3); axes[1].tick_params(axis='x', labelrotation=25)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fig_path = os.path.join(RESULTS_DIR, 'comparison_all_models_ewc.png')
    plt.savefig(fig_path, dpi=180, bbox_inches='tight')
    plt.show()

    print(f'\n=== RECAPITULATIF {DATASET_NAME} ===')
    hdr = f"{'Modele':<26} {'HF%':>7} {'BF%':>7} {'Mixte%':>9} {'Cout(CA)':>11} {'Temps(min)':>11}"
    print(hdr); print('-' * len(hdr))
    for i, name in enumerate(names):
        c = int(costs[i]) if costs[i] == costs[i] else 0
        t = times[i] if times[i] == times[i] else 0.0
        print(f"{name:<26} {acc_hf[i]:>7.2f} {acc_bf[i]:>7.2f} {acc_mixte[i]:>9.2f} "
              f"{c:>11} {t:>11.2f}")